# Building a Reproducible Mental Health Data Ecosystem: The Kilifi County, Kenya FAIR² Model Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Mental Health survey dataset for Kilifi County, Kenya, using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is described using the [Croissant schema](https://mlcommons.org/croissant/) and is accessible via the following URL:

In [ ]:
# Ensure `mlcroissant` is installed in the environment.
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and obtain an overview using `mlcroissant`. First, we instantiate the dataset from the Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'
# Instantiate the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print summary information
meta = dataset.metadata
print(f"Dataset Title: {getattr(meta, 'name', None)}")
print(f"Description: {getattr(meta, 'description', None)}")
print(f"Published: {getattr(meta, 'datePublished', None)}")
if hasattr(meta, 'keywords'):
    print(f"Keywords: {getattr(meta, 'keywords', None)}")
if hasattr(meta, 'version'):
    print(f"Version: {getattr(meta, 'version', None)}")

## 2. Data Overview
Explore the available record sets, each field and column, and their Croissant `@id`s. This is essential for referencing and extracting data—**always use the `@id` for referencing entities**.

In [ ]:
# List all record sets and their fields (referenced by @id)
print('Available record sets:')
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id} | Name: {rs.name if hasattr(rs, 'name') else ''}")
    print(f"  -> Fields: (referenced by their @id)")
    for field in rs.fields:
        print(f"     - Field @id: {field.id} | Name: {field.name} | DataType: {field.data_type if hasattr(field, 'data_type') else field.dataType}")
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"         - Column @id: {col.id} | Name: {col.name}")
    print()

## 3. Data Extraction
Load data from the record set(s) of interest into DataFrames. **Be sure to reference by the Croissant `@id`**.

> **Note:** You can find the exact `@id` for each record set from the previous cell's output. For this notebook, we will assume the primary data is in the first record set.

In [ ]:
# Select record sets to extract (by @id)
record_sets = list(dataset.record_sets)
record_set_ids = [rs.id for rs in record_sets]
# To keep references for exploration:
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display the columns (fields/columns) for the first record set
primary_record_set_id = record_set_ids[0]
print(f"Fields loaded for RecordSet @id: {primary_record_set_id}")
print(dataframes[primary_record_set_id].columns.tolist())
dataframes[primary_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We will now demonstrate basic EDA steps such as filtering, normalization, and grouping. 

Please refer to the output above for available numeric (`dataType=Float/Integer`) and categorical fields. Below, we select a numeric field and a grouping field by their `@id`.

In [ ]:
# Choose field @ids for analysis (edit as needed based on the overview output)
numeric_field_id = None  # Example: 'psq_total_score' (replace with actual field @id)
group_field_id = None    # Example: 'gender' (replace with actual field @id)

# Auto-detect a likely numeric field (first float/integer field in the primary record set)
rs = [rs for rs in dataset.record_sets if rs.id == primary_record_set_id][0]
for field in rs.fields:
    if hasattr(field, 'data_type'):
        dt = field.data_type or getattr(field, 'dataType', None)
    else:
        dt = getattr(field, 'dataType', None)
    if isinstance(dt, str) and ('Float' in dt or 'Integer' in dt):
        numeric_field_id = field.id
        break

# Auto-detect a likely group field (first string field that is likely categorical)
for field in rs.fields:
    dt = field.data_type if hasattr(field, 'data_type') else getattr(field, 'dataType', None)
    if isinstance(dt, str) and 'Text' in dt:
        # Skip the main id field
        if 'id' not in field.id.lower():
            group_field_id = field.id
            break

print(f"Selected numeric_field_id (for analysis): {numeric_field_id}")
print(f"Selected group_field_id (for groupby): {group_field_id}")

df = dataframes[primary_record_set_id]
# Check for missing fields
if numeric_field_id in df.columns and group_field_id in df.columns:
    # Remove rows with missing numeric_field values
    filtered_df = df[df[numeric_field_id].notnull()]
    # Filter with an arbitrary threshold (for demo)
    threshold = filtered_df[numeric_field_id].mean()
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}")
    print(filtered_df[[numeric_field_id, group_field_id]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id}: (first 5 rows)")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the categorical field
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Group mean of {numeric_field_id} by {group_field_id}:")
    print(grouped)
else:
    print("Could not auto-detect appropriate numeric or group fields. Modify their values using field @id from overview.")

## 5. Visualization
Let's plot the distribution of the selected numeric field and the group means using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

if 'grouped' in locals():
    plt.figure(figsize=(8,4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated the loading, exploration, and basic analysis of the FAIR² Kilifi County Mental Health dataset using `mlcroissant`, referencing all schema elements by `@id`. You can extend this notebook to deeper analyses (e.g., regression, classification, time trends) by selecting fields of interest from the overview and using their `@id` throughout your code.

**Always reference fields and record sets using their Croissant `@id` to ensure compatibility and clarity in your analyses.**